# 06 · Publication figures (style of Valavi et al. 2019, *Methods Ecol. Evol.*)

This notebook redraws the main figures in the visual style of the blockCV paper: grey panels with a white grid, bold panel titles with a subtitle and a panel letter, red block outlines with fold numbers, a terrain background clipped to Downtown, and comma-separated Easting/Northing axes. It only **reads** results that notebooks 01–05 and the R scripts wrote, and fits nothing.

| Figure | Content | Folder |
|---|---|---|
| Fig. 1 | Spatial blocks and fold numbers for the blockCV block size and its 2× and 3× sensitivity sizes | `restricted` (shows borehole locations) |
| Fig. 2 | Training and testing boreholes for one outer fold | `restricted` |
| Fig. 3 | Spatial autocorrelation range per depth slice and the block sizes | `aggregate` |
| Fig. 4 | Model comparison at the primary (blockCV) block size: every outer fold and the mean | `aggregate` |
| Fig. 5 | Robustness: Macro-F1 against block size and purge distance | `aggregate` |
| Fig. 6 | External holdout: interpolation and generalisation groups | `aggregate` |

Each figure is saved as PNG (300 dpi) and PDF (vector) in `outputs/06_figures/`.

In [ ]:
import os, sys, json
from pathlib import Path

# Run mode: "full" for the experiment, "smoke" for the reduced software validation run.
# os.environ["SIEMREAP_MODE"] = "smoke"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "project.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

from siemreap import config
cfg = config.load_config(ROOT)
MODE = config.run_mode()
print(config.smoke_banner())

import geopandas as gpd
import rasterio
from rasterio.features import geometry_mask
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter
from siemreap import geo

AGG = config.stage_dir(cfg, "06_figures", "aggregate")
RES = config.stage_dir(cfg, "06_figures", "restricted")
R04 = config.stage_dir(cfg, "04_results", "aggregate")
BCV = config.stage_dir(cfg, "04_blockcv_folds", "restricted")
SAC = config.stage_dir(cfg, "04_spatial_autocorrelation", "aggregate")
from siemreap import folds as F
blocks_m = [F.primary_block(cfg)] if MODE == "smoke" else F.block_sizes(cfg)
primary = F.primary_block(cfg)

# ggplot2 look of the blockCV paper
plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "#EBEBEB", "axes.edgecolor": "none",
    "axes.grid": True, "grid.color": "white", "grid.linewidth": 0.9, "axes.axisbelow": True,
    "xtick.color": "#4D4D4D", "ytick.color": "#4D4D4D", "xtick.labelsize": 7.5, "ytick.labelsize": 7.5,
    "xtick.major.size": 2.5, "ytick.major.size": 2.5, "axes.labelsize": 9, "axes.labelweight": "bold",
    "axes.labelcolor": "black", "legend.frameon": False, "legend.fontsize": 8, "font.family": "DejaVu Sans",
    "savefig.dpi": 300, "savefig.bbox": "tight"})
COMMA = FuncFormatter(lambda v, _: f"{v:,.0f}")
RED, BLUE, NAVY, LIGHT = "#E41A1C", "#1F3FFF", "#1F3552", "#4DA3FF"
CLAY, SAND = "#3B4CC0", "#E4572E"
FOLD_COL = ["#F8766D", "#A3A500", "#00BF7D", "#00B0F6", "#E76BF3"]   # ggplot2 hues for 5 folds
TERRAIN = LinearSegmentedColormap.from_list("blockcv", ["#7FBF3F", "#C8DC3C", "#F2E43A", "#E2B33C", "#B8762F", "#7A4A1E"])

def panel(ax, letter, title, subtitle=None):
    """Panel letter, bold title and grey subtitle above the plot, as in the blockCV figures."""
    ax.text(-0.02, 1.12 if subtitle else 1.05, f"({letter})" if letter else "", transform=ax.transAxes,
            ha="right", va="bottom", fontsize=10, fontweight="bold")
    ax.text(0.0, 1.12 if subtitle else 1.05, title, transform=ax.transAxes, ha="left", va="bottom",
            fontsize=10, fontweight="bold")
    if subtitle:
        ax.text(0.0, 1.035, subtitle, transform=ax.transAxes, ha="left", va="bottom", fontsize=7.5)

def map_axes(ax):
    ax.set_aspect("equal")
    ax.xaxis.set_major_formatter(COMMA); ax.yaxis.set_major_formatter(COMMA)
    ax.set_xlabel("Easting"); ax.set_ylabel("Northing")
    ax.locator_params(axis="x", nbins=4); ax.locator_params(axis="y", nbins=5)

def save(fig, folder, name):
    for ext in ("png", "pdf"):
        fig.savefig(folder / f"{name}.{ext}")
    print("saved", folder / f"{name}.png")

# shared inputs: Downtown polygon, DEM clipped to it, development boreholes (measured cells only)
poly = geo.load_polygon(cfg, "downtown_shp")
with rasterio.open(cfg["_root"] / cfg["sources"]["dem"]) as r:
    step = max(1, max(r.width, r.height) // 900)
    dem = r.read(1, out_shape=(r.height // step, r.width // step)).astype(float)
    tr = r.transform * r.transform.scale(r.width / dem.shape[1], r.height / dem.shape[0])
    nodata = r.nodata
dem[dem == nodata] = np.nan
dem[geometry_mask([poly], out_shape=dem.shape, transform=tr)] = np.nan
dem_extent = (tr.c, tr.c + tr.a * dem.shape[1], tr.f + tr.e * dem.shape[0], tr.f)

mt = pd.read_csv(config.processed_dir(cfg) / "cells_all.csv")
meas = mt[(mt.measured == 1) if "measured" in mt else True]
bh = (meas[meas.in_A].groupby("BH_Code")
        .agg(E=("UTM_Easting_m", "first"), N=("UTM_Northing_m", "first"), clay_share=("y", "mean")))
bh["dominant"] = np.where(bh.clay_share >= 0.5, "Clay", "Sand")
xmin, ymin, xmax, ymax = poly.bounds
pad = 150
print(len(bh), "Downtown development boreholes; DEM grid", dem.shape)

def background(ax):
    ax.imshow(dem, extent=dem_extent, cmap=TERRAIN, origin="upper", interpolation="nearest", zorder=1)
    gpd.GeoSeries([poly]).boundary.plot(ax=ax, color="#4D4D4D", linewidth=0.5, zorder=2)
    ax.set_xlim(xmin - pad, xmax + pad); ax.set_ylim(ymin - pad, ymax + pad)

## Fig. 1 · Spatial blocks and fold assignment

In [ ]:
fig, axes = plt.subplots(1, len(blocks_m), figsize=(5.2 * len(blocks_m), 5.2), squeeze=False)
for ax, letter, b in zip(axes[0], "abc", blocks_m):
    blk = gpd.read_file(BCV / f"blocks_{b}m.gpkg")
    background(ax)
    # blocks are small next to the Downtown extent, so folds are shown by fill colour instead of numbers
    for f, g in blk.dropna(subset=["folds"]).groupby("folds"):
        g.plot(ax=ax, facecolor=FOLD_COL[int(f) - 1], edgecolor=RED, linewidth=0.4, zorder=3)
    map_axes(ax)
    panel(ax, letter, "Spatial blocks", f"{b} m square blocks, k = {cfg['blockcv']['k']}, random fold assignment")
axes[0][-1].legend(handles=[Patch(facecolor=c, edgecolor=RED, label=f"Fold {i + 1}") for i, c in enumerate(FOLD_COL)],
                   loc="center left", bbox_to_anchor=(1.02, 0.5))
fig.tight_layout()
save(fig, RES, "fig1_spatial_blocks")

## Fig. 2 · Training and testing boreholes for one outer fold

In [ ]:
FOLD = 1                                     # outer fold to illustrate
folds = pd.read_csv(config.folds_dir(cfg) / f"outer_folds_{primary}m.csv").set_index("BH_Code")
blk = gpd.read_file(BCV / f"blocks_{primary}m.gpkg")
b_fold = bh.join(folds.fold, how="inner")
sets = {"Training set": b_fold[b_fold.fold != FOLD], "Testing set": b_fold[b_fold.fold == FOLD]}

fig, axes = plt.subplots(1, 3, figsize=(15.5, 5.2))
ax = axes[0]
background(ax)
blk.boundary.plot(ax=ax, color=RED, linewidth=0.5, zorder=3)
blk[blk.folds == FOLD].plot(ax=ax, facecolor=BLUE, edgecolor=BLUE, linewidth=1.2, zorder=4)
map_axes(ax)
panel(ax, "a", "Spatial blocks", f"{primary} m blocks; test blocks of fold {FOLD} in blue")
for ax, letter, (name, d) in zip(axes[1:], "bc", sets.items()):
    background(ax)
    blk.boundary.plot(ax=ax, color=RED, linewidth=0.4, zorder=3)
    for cls, col in (("Sand", SAND), ("Clay", CLAY)):
        s = d[d.dominant == cls]
        ax.scatter(s.E, s.N, s=16, color=col, edgecolor="white", linewidth=0.3, zorder=5)
    map_axes(ax)
    panel(ax, letter, name, f"{len(d)} boreholes")
axes[2].legend(handles=[Line2D([], [], marker="o", ls="", color=CLAY, label="Clay-dominated"),
                        Line2D([], [], marker="o", ls="", color=SAND, label="Sand-dominated")],
               title="Borehole (0–15 m)", title_fontproperties={"weight": "bold", "size": 8},
               loc="center left", bbox_to_anchor=(1.02, 0.5))
fig.tight_layout()
save(fig, RES, f"fig2_train_test_fold{FOLD}")

## Fig. 3 · Spatial autocorrelation range and block size

In [ ]:
auto = pd.read_csv(SAC / "blockcv_spatial_autocor_by_slice.csv")
vg = pd.read_csv(SAC / "variogram_parameters.csv")
vg = vg[~vg.slice.str.startswith("pooled")]

def range_bars(ax, slices, ranges, highlight=None, hatch_mask=None):
    cols = [LIGHT if (highlight is not None and i == highlight) else NAVY for i in range(len(ranges))]
    bars = ax.bar(range(len(ranges)), ranges, color=cols, width=0.7, zorder=3)
    if hatch_mask is not None:
        for bar, h in zip(bars, hatch_mask):
            if h:
                bar.set_color("#9E9E9E")
    ax.set_xticks(range(len(slices)), slices, rotation=90)
    ax.set_xlabel("Depth slice"); ax.set_ylabel("Ranges (m)")
    for b in blocks_m:
        ax.axhline(b, color=RED if b == primary else "#4D4D4D", linestyle="--" if b == primary else ":",
                   linewidth=1.2 if b == primary else 0.9, zorder=4)
        ax.text(-0.4, b, f"Block size {b} m", color=RED if b == primary else "#4D4D4D",
                ha="left", va="bottom", fontsize=7.5)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
if auto.range_parameter_m.notna().any():
    r_ = auto.range_parameter_m.values
    range_bars(axes[0], auto.slice, r_)
    panel(axes[0], "a", "Autocorrelation range (blockCV)",
          "cv_spatial_autocor range per 1.5 m slice; red dashed = their median = block size")
else:
    axes[0].text(0.5, 0.5, "cv_spatial_autocor not available", transform=axes[0].transAxes, ha="center")
    axes[0].set_xticks([]); axes[0].set_yticks([])
    panel(axes[0], "a", "Autocorrelation range", "blockCV cv_spatial_autocor")
unresolved = ~vg.resolved.astype(str).str.upper().eq("TRUE")
range_bars(axes[1], vg.slice, vg.practical_range_m.values, hatch_mask=unresolved.values)
axes[1].legend(handles=[Patch(color=NAVY, label="Resolved fit"), Patch(color="#9E9E9E", label="Unresolved (range < first lag)")],
               loc="upper left", fontsize=7.5)
panel(axes[1], "b", "Variogram range", "gstat indicator variograms, practical range per 1.5 m slice")
fig.tight_layout()
save(fig, AGG, "fig3_autocorrelation_range")

## Fig. 4 · Model comparison at the primary block size

In [ ]:
folds_t = pd.read_csv(R04 / "metrics_by_outer_fold.csv")
order = [m for m in ["DEPTH_PRIOR", "RF", "XGB", "MLP", "GWRF", "GWXGB", "GP"] if m in set(folds_t.model)]
label = {"DEPTH_PRIOR": "Depth only"}
hue = ["#F8766D", "#B79F00", "#00BA38", "#00BFC4", "#619CFF", "#F564E3"]   # ggplot2 default hues
colour = {"DEPTH_PRIOR": "#7F7F7F", **{m: hue[i % len(hue)] for i, m in enumerate(o for o in order if o != "DEPTH_PRIOR")}}
main = folds_t[(folds_t.block_m == primary) & (folds_t.purge_m == 0)]
scen = sorted(main.scenario.unique())
title = {"A": "Scenario A", "B": "Scenario B"}
sub = {"A": "Trained on Downtown boreholes only", "B": "Trained on Downtown + 5 km ring boreholes"}
fig, axes = plt.subplots(1, len(scen), figsize=(5.6 * len(scen), 4.4), sharey=True, squeeze=False)
rng = np.random.default_rng(1)
for ax, letter, sc in zip(axes[0], "ab", scen):
    d = main[main.scenario == sc]
    ref = d[d.model == "DEPTH_PRIOR"].macro_f1.mean()
    ax.axhline(ref, color="#7F7F7F", linestyle="--", linewidth=1, zorder=2)
    ax.axhline(0.5, color="black", linestyle=":", linewidth=0.8, zorder=2)
    for i, m in enumerate(order):
        v = d[d.model == m].macro_f1
        ax.scatter(i + rng.uniform(-0.12, 0.12, len(v)), v, s=22, color=colour[m], alpha=0.75,
                   edgecolor="white", linewidth=0.4, zorder=3)
        ax.errorbar(i, v.mean(), yerr=v.std(), fmt="D", color="black", markersize=5, capsize=3, linewidth=1, zorder=4)
    ax.set_xticks(range(len(order)), [label.get(m, m) for m in order])
    ax.set_xlabel("Model"); ax.set_ylabel("Macro-F1 (outer test folds)")
    panel(ax, letter, title.get(sc, sc), f"{sub.get(sc, '')}; {primary} m blocks; dots = folds, ◆ = mean ± SD")
axes[0][-1].legend(handles=[Line2D([], [], color="#7F7F7F", ls="--", label="Depth-only mean"),
                            Line2D([], [], color="black", ls=":", label="Random (0.5)")],
                   loc="center left", bbox_to_anchor=(1.02, 0.5))
fig.tight_layout()
save(fig, AGG, "fig4_model_comparison")

## Fig. 5 · Robustness to block size and spatial purge

In [ ]:
summ = pd.read_csv(R04 / "metrics_fold_mean_sd.csv")
rob_models = [m for m in ["DEPTH_PRIOR", "RF", "XGB"] if m in set(summ.model)]
fig, axes = plt.subplots(2, len(scen), figsize=(5.6 * len(scen), 8.2), sharey=True, squeeze=False)
for row, (key, xs, xlab, sub0) in enumerate([
        ("block_m", sorted(summ.block_m.unique()), "Block size (m)", "outer folds from blockCV blocks, no purge"),
        ("purge_m", sorted(summ[summ.block_m == primary].purge_m.unique()), "Purge distance (m)",
         f"{primary} m blocks; training boreholes within the distance removed")]):
    for col, sc in enumerate(scen):
        ax = axes[row][col]
        d = summ[(summ.scenario == sc) & (summ.purge_m == 0 if key == "block_m" else summ.block_m == primary)]
        for m in rob_models:
            s = d[d.model == m].set_index(key).reindex(xs)
            if s.macro_f1_mean.notna().sum() == 0:
                continue
            ax.errorbar(range(len(xs)), s.macro_f1_mean, yerr=s.macro_f1_sd.fillna(0), color=colour[m],
                        marker="o", markersize=5, capsize=3, linewidth=1.4,
                        linestyle="--" if m == "DEPTH_PRIOR" else "-", label=label.get(m, m), zorder=3)
        ax.axhline(0.5, color="black", linestyle=":", linewidth=0.8, zorder=2)
        ax.set_xticks(range(len(xs)), [f"{x:g}" for x in xs])
        ax.set_xlabel(xlab); ax.set_ylabel("Macro-F1 (outer test folds)")
        panel(ax, "abcd"[row * len(scen) + col], f"{title.get(sc, sc)}: {'block size' if row == 0 else 'spatial purge'}", sub0)
axes[0][-1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5), title="Model",
                   title_fontproperties={"weight": "bold", "size": 8})
fig.tight_layout()
save(fig, AGG, "fig5_robustness")

## Fig. 6 · External holdout

In [ ]:
ext_path = config.stage_dir(cfg, "05_final", "aggregate") / "external_evaluation.json"
ext_rec = json.loads(ext_path.read_text())
ext = pd.DataFrame(ext_rec["metrics"])
if "model" in ext:   # revision 3: final model on all cells (measured + filled); the depth-only reference is in notebook 05
    ext = ext[(ext.model != "DEPTH_PRIOR") & (ext.cells == "all cells")]
ext = ext.set_index("group")
groups = [g for g in ["all", "interpolation", "generalisation", "smoke_stand_in"] if g in ext.index]
names = {"all": "All", "interpolation": "Interpolation", "generalisation": "Generalisation", "smoke_stand_in": "Stand-in (smoke)"}
metrics = [("macro_f1", "Macro-F1"), ("accuracy", "Accuracy"), ("roc_auc", "ROC-AUC")]
fig, ax = plt.subplots(figsize=(7.2, 4.4))
w = 0.8 / len(metrics)
for j, (k, lab) in enumerate(metrics):
    ax.bar(np.arange(len(groups)) + (j - (len(metrics) - 1) / 2) * w, ext.loc[groups, k], width=w * 0.95,
           color=hue[[0, 2, 4][j]], label=lab, zorder=3)
ax.axhline(0.5, color="black", linestyle=":", linewidth=0.8, zorder=4)
ax.set_xticks(range(len(groups)), [f"{names[g]}\n(n = {int(ext.loc[g, 'n'])} cells)" for g in groups])
ax.set_ylim(0, 1); ax.set_ylabel("Score"); ax.set_xlabel("External boreholes")
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), title="Metric", title_fontproperties={"weight": "bold", "size": 8})
panel(ax, "", "External holdout", f"Final model, evaluated once ({ext_rec['label'].split('(')[0].strip().lower()})")
fig.tight_layout()
save(fig, AGG, "fig6_external_holdout")

**Notes for the thesis.** Figs. 1 and 2 show borehole locations and are saved under `restricted/`; check confidentiality before publishing them. In Figs. 4–6 the dotted line marks random guessing (0.5) and the dashed grey line the depth-only reference. Figure size, fonts and colours are set in the first code cell.